# Building an AI Classifier: Identifying Cats, Dogs & Pandas with PyTorch

This notebook fine-tunes a pre-trained CNN (ResNet18) to classify images into
**cat**, **dog**, or **panda** using transfer learning, following the workshop
requirements:

1. Environment setup (CUDA check)
2. Data preparation (Kaggle dataset, `ImageFolder`, transforms/augmentation)
3. Model design (pretrained backbone frozen, custom classifier head)
4. Training (10-15 epochs, checkpointing best model)
5. Evaluation (test loss, accuracy, confusion matrix, example predictions)
6. Bonus: single-image prediction function + optional Streamlit app


## 1. Environment Setup

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


**Notes**

- **Running locally:** you need an NVIDIA GPU with CUDA Toolkit + drivers installed.
  See the official [PyTorch installation guide](https://pytorch.org/get-started/locally/)
  to install the correct `torch`/`torchvision` build for your CUDA version.
- **Running on Kaggle:** go to *Settings → Accelerator → GPU* to enable a GPU runtime
  before running this notebook.


In [ ]:
# Uncomment if torch/torchvision/torchaudio are not already installed in your environment
# %pip install torch torchvision torchaudio --upgrade


In [ ]:
import os
import time
import copy
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, models, transforms

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

SEED = 33
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


## 2. Data Preparation

### 2.1 Download the dataset

The [Cats vs Dogs vs Pandas dataset](https://www.kaggle.com/datasets/gpiosenka/cats-dogs-pandas-images)
can be downloaded directly inside the notebook using the Kaggle CLI. This requires
a `kaggle.json` API token placed at `~/.kaggle/kaggle.json` (or uploaded to the
Kaggle/Colab environment).


In [ ]:
# Requires the Kaggle API token to be configured (kaggle.json)
# !pip install kaggle --quiet
# !mkdir -p ./data
# !kaggle datasets download -d gpiosenka/cats-dogs-pandas-images -p ./data --unzip


### 2.2 Expected folder structure

```
data/
├── train/
│   ├── cat/
│   ├── dog/
│   └── panda/
└── test/
    ├── cat/
    ├── dog/
    └── panda/
```

If the downloaded dataset uses different folder names (e.g. `Cat`, `Dog`, `Panda`,
or a single folder that needs splitting), reorganize it into the `train/` and
`test/` structure above before continuing. Adjust `DATA_DIR` below if needed.


In [ ]:
DATA_DIR = "./data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

CLASS_NAMES = ["cat", "dog", "panda"]
IMG_SIZE = 224
BATCH_SIZE = 32

# ImageNet normalization stats (required when using ImageNet-pretrained backbones)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


In [ ]:
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Classes found:", train_dataset.classes)
print("Train size:", len(train_dataset))
print("Test size :", len(test_dataset))


In [ ]:
# Visualize a batch of training images
def imshow(inp, title=None):
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array(IMAGENET_MEAN)
    std = np.array(IMAGENET_STD)
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.axis("off")

images, labels = next(iter(train_loader))
grid = torchvision.utils.make_grid(images[:8])
plt.figure(figsize=(12, 4))
imshow(grid, title=[train_dataset.classes[l] for l in labels[:8]])
plt.show()


## 3. Model Design

- Backbone: pretrained **ResNet18** (ImageNet weights).
- All convolutional layers are **frozen** (no gradient updates).
- The final fully-connected layer is replaced with a custom classifier head:
  `Linear(in_features, 256) -> ReLU -> Dropout(p=0.5) -> Linear(256, 3)`.


In [ ]:
def build_model(num_classes=3, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.5),
        nn.Linear(256, num_classes),
    )
    return model

model = build_model(num_classes=len(CLASS_NAMES), freeze_backbone=True)
model = model.to(device)
print(model.fc)


## 4. Training

Only the parameters of the new classifier head (`model.fc`) require gradients,
since the backbone is frozen — Adam only optimizes those parameters.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

EPOCHS = 15
BEST_MODEL_PATH = "best_model.pth"


In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy, all_preds, all_labels


In [ ]:
best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
history = {"train_loss": [], "val_loss": [], "val_acc": []}

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_dataset)
    val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch [{epoch}/{EPOCHS}]  "
          f"Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%")

    # Save best model checkpoint based on validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, BEST_MODEL_PATH)

print(f"\nTraining complete in {time.time() - start_time:.0f} seconds")
print(f"Best validation accuracy: {best_val_acc:.2f}%")

# Load best weights back into the model
model.load_state_dict(best_model_wts)


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss over Epochs")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["val_acc"], label="Val Accuracy", color="green")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy over Epochs")
plt.legend()

plt.tight_layout()
plt.show()


## 5. Evaluation

Report the final test loss and accuracy using the best saved checkpoint, plus a
confusion matrix and a few example predictions.


In [ ]:
test_loss, test_acc, all_preds, all_labels = evaluate(model, test_loader, criterion, device)

print("Name: <YOUR NAME>")
print("Register No: <YOUR REGISTER NUMBER>")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix")
plt.show()


In [ ]:
# Show a few example predictions
def visualize_predictions(model, loader, class_names, device, num_images=8):
    model.eval()
    images_shown = 0
    plt.figure(figsize=(14, 6))

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for j in range(inputs.size(0)):
                if images_shown >= num_images:
                    plt.tight_layout()
                    plt.show()
                    return
                images_shown += 1
                ax = plt.subplot(2, num_images // 2, images_shown)
                ax.axis("off")
                actual = class_names[labels[j]]
                predicted = class_names[preds[j]]
                color = "green" if actual == predicted else "red"
                ax.set_title(f"pred: {predicted}\ntrue: {actual}", color=color)
                imshow(inputs.cpu().data[j])

    plt.tight_layout()
    plt.show()

visualize_predictions(model, test_loader, CLASS_NAMES, device, num_images=8)


## 6. BONUS: Predict a new uploaded image

The function below loads a single image file, applies the same preprocessing
used for the test set, and returns the predicted class (cat / dog / panda)
along with class probabilities.


In [ ]:
import torch.nn.functional as F
from PIL import Image

def predict_image(image_path, model=model, class_names=CLASS_NAMES, device=device):
    """
    Classify a single image as 'cat', 'dog', or 'panda'.

    image_path : path to a local image file (jpg/png/etc.)
    """
    image = Image.open(image_path).convert("RGB")
    input_tensor = test_transforms(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(input_tensor)
        probs = F.softmax(logits, dim=1)
        pred_idx = logits.argmax(dim=1).item()

    pred_class = class_names[pred_idx]
    print("Name: <YOUR NAME>")
    print("Register No: <YOUR REGISTER NUMBER>")
    print(f"Predicted class: {pred_class}")
    for cls, p in zip(class_names, probs[0].cpu().numpy()):
        print(f"  P({cls}) = {p:.4f}")

    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Prediction: {pred_class}")
    plt.show()

    return pred_class, probs.cpu().numpy()

# Example usage:
# predict_image("./data/test/cat/some_cat_image.jpg")


### Optional: Deploy with Streamlit

A minimal Streamlit app for interactive image upload + prediction. Save this as
`app.py` in the repository root and run with `streamlit run app.py` (requires
`best_model.pth` from training to be present in the same directory).

```python
import streamlit as st
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image

CLASS_NAMES = ["cat", "dog", "panda"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

@st.cache_resource
def load_model():
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.5),
        nn.Linear(256, len(CLASS_NAMES)),
    )
    model.load_state_dict(torch.load("best_model.pth", map_location="cpu"))
    model.eval()
    return model

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

st.title("Cat / Dog / Panda Classifier")
uploaded_file = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert("RGB")
    st.image(image, caption="Uploaded image", use_column_width=True)

    model = load_model()
    input_tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        logits = model(input_tensor)
        probs = F.softmax(logits, dim=1)
        pred_idx = logits.argmax(dim=1).item()

    st.write(f"**Prediction:** {CLASS_NAMES[pred_idx]}")
    for cls, p in zip(CLASS_NAMES, probs[0].numpy()):
        st.write(f"{cls}: {p:.4f}")
```


## RESULT

A transfer-learning image classifier (ResNet18 backbone, frozen convolutional
layers, custom fully-connected head with 256 neurons and dropout p=0.5) was
trained successfully to distinguish between cats, dogs, and pandas. Test loss,
accuracy, a confusion matrix, and example predictions are reported above, and
the best-performing checkpoint was saved to `best_model.pth`.
